In [7]:
import matplotlib.pyplot as plt

# You may change the mhealth_activity module but your algorithm must support the original version
from mhealth_activity import Recording, Trace, Activity, WatchLocation, Path

# For interactive plots, uncomment the following line
# %matplotlib widget

In [13]:
import os
import numpy as np
import pandas as pd
from scipy.signal import find_peaks
from scipy.stats import skew, kurtosis
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import ExtraTreesClassifier
import joblib


# ============================================================
# Trace helpers
# ============================================================

def trace_values(recording, key):
    if key not in recording.data:
        return np.array([], dtype=float)

    tr = recording.data[key]

    for attr in ["values", "data", "y", "v"]:
        if hasattr(tr, attr):
            arr = np.asarray(getattr(tr, attr), dtype=float)
            return arr[np.isfinite(arr)]

    try:
        arr = np.asarray(tr, dtype=float)
        return arr[np.isfinite(arr)]
    except Exception:
        raise ValueError(
            f"Could not extract numeric values from {key}. "
            f"Run dir(recording.data['{key}']) to inspect it."
        )


def trace_total_time(recording, key="timestamp"):
    if key not in recording.data:
        return np.nan
    return getattr(recording.data[key], "total_time", np.nan)


def magnitude(x, y, z):
    n = min(len(x), len(y), len(z))
    if n == 0:
        return np.array([], dtype=float)
    return np.sqrt(x[:n] ** 2 + y[:n] ** 2 + z[:n] ** 2)


# ============================================================
# Feature helpers
# ============================================================

def basic_stats(x, prefix):
    if len(x) < 5:
        return {
            f"{prefix}_mean": np.nan,
            f"{prefix}_std": np.nan,
            f"{prefix}_min": np.nan,
            f"{prefix}_max": np.nan,
            f"{prefix}_range": np.nan,
            f"{prefix}_median": np.nan,
            f"{prefix}_iqr": np.nan,
            f"{prefix}_skew": np.nan,
            f"{prefix}_kurt": np.nan,
        }

    return {
        f"{prefix}_mean": np.mean(x),
        f"{prefix}_std": np.std(x),
        f"{prefix}_min": np.min(x),
        f"{prefix}_max": np.max(x),
        f"{prefix}_range": np.max(x) - np.min(x),
        f"{prefix}_median": np.median(x),
        f"{prefix}_iqr": np.percentile(x, 75) - np.percentile(x, 25),
        f"{prefix}_skew": skew(x),
        f"{prefix}_kurt": kurtosis(x),
    }


def peak_features(x, prefix):
    if len(x) < 20:
        return {
            f"{prefix}_n_peaks": np.nan,
            f"{prefix}_peak_rate": np.nan,
            f"{prefix}_peak_prom_mean": np.nan,
        }

    x_centered = x - np.median(x)
    prom = max(np.std(x_centered), 1e-6)

    peaks, props = find_peaks(
        x_centered,
        prominence=prom,
        distance=5
    )

    if len(peaks) == 0:
        return {
            f"{prefix}_n_peaks": 0,
            f"{prefix}_peak_rate": 0,
            f"{prefix}_peak_prom_mean": 0,
        }

    return {
        f"{prefix}_n_peaks": len(peaks),
        f"{prefix}_peak_rate": len(peaks) / len(x),
        f"{prefix}_peak_prom_mean": np.mean(props["prominences"]),
    }


def segment_stats(x, prefix, n_segments=4):
    feats = {}

    for i in range(n_segments):
        feats[f"{prefix}_seg{i}_mean"] = np.nan
        feats[f"{prefix}_seg{i}_std"] = np.nan
        feats[f"{prefix}_seg{i}_range"] = np.nan
        feats[f"{prefix}_seg{i}_median"] = np.nan

    if len(x) < n_segments * 5:
        return feats

    segments = np.array_split(x, n_segments)

    for i, seg in enumerate(segments):
        feats[f"{prefix}_seg{i}_mean"] = np.mean(seg)
        feats[f"{prefix}_seg{i}_std"] = np.std(seg)
        feats[f"{prefix}_seg{i}_range"] = np.max(seg) - np.min(seg)
        feats[f"{prefix}_seg{i}_median"] = np.median(seg)

    return feats


def burst_features(x, prefix):
    if len(x) < 20:
        return {
            f"{prefix}_burst_count": np.nan,
            f"{prefix}_burst_ratio": np.nan,
            f"{prefix}_burst_mean_value": np.nan,
        }

    threshold = np.percentile(x, 90)
    burst_mask = x > threshold

    burst_count = np.sum(np.diff(burst_mask.astype(int)) == 1)

    return {
        f"{prefix}_burst_count": burst_count,
        f"{prefix}_burst_ratio": np.mean(burst_mask),
        f"{prefix}_burst_mean_value": np.mean(x[burst_mask]) if np.any(burst_mask) else 0,
    }


def heading_array(mx, my):
    n = min(len(mx), len(my))
    if n < 10:
        return np.array([], dtype=float)

    return np.unwrap(np.arctan2(my[:n], mx[:n]))


def heading_proxy_features(mx, my):
    heading = heading_array(mx, my)

    if len(heading) < 10:
        return {
            "heading_total_abs_change": np.nan,
            "heading_net_change": np.nan,
            "heading_std": np.nan,
            "heading_large_turns": np.nan,
            "heading_start": np.nan,
            "heading_end": np.nan,
        }

    diff = np.diff(heading)

    return {
        "heading_total_abs_change": np.sum(np.abs(diff)),
        "heading_net_change": heading[-1] - heading[0],
        "heading_std": np.std(heading),
        "heading_large_turns": np.sum(np.abs(diff) > np.percentile(np.abs(diff), 95)),
        "heading_start": heading[0],
        "heading_end": heading[-1],
    }


# ============================================================
# Feature extraction
# Uses:
# - duration
# - altitude / is_uphill
# - gyroscope
# - magnetometer
# - heading proxy
# Does NOT use:
# - accelerometer
# - pressure
# - GPS longitude/latitude/bearing/speed
# - phone steps
# ============================================================

def extract_path_features(recording):
    features = {}

    features["duration_sec"] = trace_total_time(recording, "timestamp")

    # --------------------------------------------------------
    # Altitude direction
    # --------------------------------------------------------
    altitude = trace_values(recording, "altitude")

    if len(altitude) >= 10:
        initial_alt = np.median(altitude[:10])
        final_alt = np.median(altitude[-10:])
        alt_delta = final_alt - initial_alt

        features["alt_initial"] = initial_alt
        features["alt_final"] = final_alt
        features["alt_delta"] = alt_delta
        features["is_uphill"] = int(alt_delta > 0)
    else:
        features["alt_initial"] = np.nan
        features["alt_final"] = np.nan
        features["alt_delta"] = np.nan
        features["is_uphill"] = np.nan

    features.update(segment_stats(altitude, "altitude", n_segments=4))

    # --------------------------------------------------------
    # Gyroscope
    # --------------------------------------------------------
    gx = trace_values(recording, "gx")
    gy = trace_values(recording, "gy")
    gz = trace_values(recording, "gz")

    gyro_mag = magnitude(gx, gy, gz)

    features.update(basic_stats(gyro_mag, "gyro_mag"))
    features.update(peak_features(gyro_mag, "gyro_mag"))
    features.update(burst_features(gyro_mag, "gyro_mag"))
    features.update(segment_stats(gyro_mag, "gyro_mag", n_segments=4))

    features.update(basic_stats(gx, "gx"))
    features.update(basic_stats(gy, "gy"))
    features.update(basic_stats(gz, "gz"))

    features.update(segment_stats(gx, "gx", n_segments=4))
    features.update(segment_stats(gy, "gy", n_segments=4))
    features.update(segment_stats(gz, "gz", n_segments=4))

    features["gx_total_abs_change"] = np.sum(np.abs(np.diff(gx))) if len(gx) > 5 else np.nan
    features["gy_total_abs_change"] = np.sum(np.abs(np.diff(gy))) if len(gy) > 5 else np.nan
    features["gz_total_abs_change"] = np.sum(np.abs(np.diff(gz))) if len(gz) > 5 else np.nan

    features["gx_net_change"] = gx[-1] - gx[0] if len(gx) > 5 else np.nan
    features["gy_net_change"] = gy[-1] - gy[0] if len(gy) > 5 else np.nan
    features["gz_net_change"] = gz[-1] - gz[0] if len(gz) > 5 else np.nan

    # --------------------------------------------------------
    # Magnetometer + heading proxy
    # --------------------------------------------------------
    mx = trace_values(recording, "mx")
    my = trace_values(recording, "my")
    mz = trace_values(recording, "mz")

    mag_mag = magnitude(mx, my, mz)
    heading = heading_array(mx, my)

    features.update(basic_stats(mag_mag, "mag_mag"))
    features.update(segment_stats(mag_mag, "mag_mag", n_segments=4))

    features.update(basic_stats(mx, "mx"))
    features.update(basic_stats(my, "my"))
    features.update(basic_stats(mz, "mz"))

    features.update(segment_stats(mx, "mx", n_segments=4))
    features.update(segment_stats(my, "my", n_segments=4))
    features.update(segment_stats(mz, "mz", n_segments=4))

    features.update(heading_proxy_features(mx, my))
    features.update(segment_stats(heading, "heading", n_segments=4))
    features.update(burst_features(np.abs(np.diff(heading)), "heading_diff"))

    return features


# ============================================================
# Two-stage path classifier
# ============================================================

class TwoStagePathClassifier:
    def __init__(self):
        self.direction_model = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", ExtraTreesClassifier(
                n_estimators=500,
                random_state=1,
                class_weight="balanced",
                n_jobs=-1
            ))
        ])

        self.uphill_model = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", ExtraTreesClassifier(
                n_estimators=700,
                random_state=2,
                class_weight="balanced",
                n_jobs=-1
            ))
        ])

        self.downhill_model = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", ExtraTreesClassifier(
                n_estimators=700,
                random_state=3,
                class_weight="balanced",
                n_jobs=-1
            ))
        ])

        self.feature_columns = None

    def fit(self, X, y):
        self.feature_columns = list(X.columns)

        y_direction = np.isin(y, [0, 1, 2]).astype(int)
        self.direction_model.fit(X, y_direction)

        uphill_mask = np.isin(y, [0, 1, 2])
        downhill_mask = np.isin(y, [3, 4])

        self.uphill_model.fit(X[uphill_mask], y[uphill_mask])
        self.downhill_model.fit(X[downhill_mask], y[downhill_mask])

        return self

    def predict(self, X):
        X = X[self.feature_columns]

        direction_pred = self.direction_model.predict(X)
        final_preds = []

        for i in range(len(X)):
            row = X.iloc[[i]]

            if direction_pred[i] == 1:
                final_preds.append(self.uphill_model.predict(row)[0])
            else:
                final_preds.append(self.downhill_model.predict(row)[0])

        return np.array(final_preds)


# ============================================================
# Load recordings
# ============================================================

TRAIN_DIR = "data/train"

recordings = []
path_labels = []
filenames = []

for filename in sorted(os.listdir(TRAIN_DIR)):
    if not filename.endswith(".pkl"):
        continue

    filepath = os.path.join(TRAIN_DIR, filename)
    rec = Recording(filepath)

    if rec.labels is None:
        continue

    recordings.append(rec)
    path_labels.append(rec.labels["path_idx"])
    filenames.append(filename)

y = np.array(path_labels)

print("Loaded recordings:", len(recordings))
print("\nPath label counts:")
print(pd.Series(y).value_counts().sort_index())


# ============================================================
# Extract features
# ============================================================

feature_rows = []

for rec in recordings:
    feature_rows.append(extract_path_features(rec))

X = pd.DataFrame(feature_rows)

print("\nFeature matrix shape:", X.shape)
print("\nNumber of features:", len(X.columns))


# ============================================================
# Debug uphill feature
# ============================================================

debug_df = X.copy()
debug_df["path_idx"] = y
debug_df["filename"] = filenames

print("\nAltitude direction by path:")
print(debug_df.groupby("path_idx")[["alt_delta", "is_uphill"]].mean())


# ============================================================
# Train / validation split
# ============================================================

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# ============================================================
# Train two-stage model
# ============================================================

model = TwoStagePathClassifier()
model.fit(X_train, y_train)


# ============================================================
# Evaluate full path classifier
# ============================================================

y_pred = model.predict(X_val)

print("\nFull path validation accuracy:")
print(accuracy_score(y_val, y_pred))

print("\nFull path classification report:")
print(classification_report(y_val, y_pred))

print("\nFull path confusion matrix:")
print(confusion_matrix(y_val, y_pred))


# ============================================================
# Evaluate uphill/downhill stage separately
# ============================================================

y_direction_true = np.isin(y_val, [0, 1, 2]).astype(int)
y_direction_pred = model.direction_model.predict(X_val)

print("\nUphill/downhill accuracy:")
print(accuracy_score(y_direction_true, y_direction_pred))

print("\nUphill/downhill confusion matrix:")
print(confusion_matrix(y_direction_true, y_direction_pred))


# ============================================================
# Evaluate within-group only
# ============================================================

uphill_val_mask = np.isin(y_val, [0, 1, 2])
downhill_val_mask = np.isin(y_val, [3, 4])

if uphill_val_mask.sum() > 0:
    uphill_pred = model.uphill_model.predict(X_val[uphill_val_mask])
    print("\nUphill-only accuracy, paths 0/1/2:")
    print(accuracy_score(y_val[uphill_val_mask], uphill_pred))
    print(confusion_matrix(y_val[uphill_val_mask], uphill_pred))

if downhill_val_mask.sum() > 0:
    downhill_pred = model.downhill_model.predict(X_val[downhill_val_mask])
    print("\nDownhill-only accuracy, paths 3/4:")
    print(accuracy_score(y_val[downhill_val_mask], downhill_pred))
    print(confusion_matrix(y_val[downhill_val_mask], downhill_pred))


# ============================================================
# Feature importances for subgroup models
# ============================================================

uphill_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model.uphill_model.named_steps["clf"].feature_importances_
}).sort_values("importance", ascending=False)

downhill_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model.downhill_model.named_steps["clf"].feature_importances_
}).sort_values("importance", ascending=False)

print("\nTop 25 uphill model features:")
print(uphill_importance.head(25))

print("\nTop 25 downhill model features:")
print(downhill_importance.head(25))


# ============================================================
# Retrain on all data and save
# ============================================================

final_model = TwoStagePathClassifier()
final_model.fit(X, y)

joblib.dump(final_model, "groupXX_path_model.joblib")

print("\nSaved final model as groupXX_path_model.joblib")


# ============================================================
# Single-recording prediction example
# ============================================================

# test_rec = Recording("data/test/test_trace_001.pkl")
# X_one = pd.DataFrame([extract_path_features(test_rec)])
# predicted_path = int(final_model.predict(X_one)[0])
# print("Predicted path:", predicted_path)

Loaded recordings: 396

Path label counts:
0    82
1    76
2    74
3    71
4    93
Name: count, dtype: int64

Feature matrix shape: (396, 258)

Number of features: 258

Altitude direction by path:
          alt_delta  is_uphill
path_idx                      
0         42.070909   0.951220
1         40.432325   0.960526
2         49.026660   0.986486
3        -51.295879   0.014085
4        -44.035477   0.043011

Full path validation accuracy:
0.7375

Full path classification report:
              precision    recall  f1-score   support

           0       0.53      0.47      0.50        17
           1       0.72      0.87      0.79        15
           2       0.53      0.53      0.53        15
           3       0.93      0.93      0.93        14
           4       0.94      0.89      0.92        19

    accuracy                           0.74        80
   macro avg       0.73      0.74      0.73        80
weighted avg       0.74      0.74      0.73        80


Full path confusion mat